# Reddit Community Intelligence (RCI v6.1.0) — Executable Prototype Walkthrough (Deliverable 3)

This notebook walks end-to-end through the **100% real, strictly inductive, zero-leakage RCI v6.1.0 prototype**:
1. Real anonymized Reddit corpus overview (`365` threads, `3,557` comments, `0` PII, `0` synthetic rows)
2. **Task 5** Indian-Reddit Language & Script ID vs. `langdetect` failure study
3. **Task 6** Gen-Z Slang & Emoji semantic affect normalization + 5-fold OOF preprocessing ablation
4. **Tasks 7–9** Unsupervised 10-cluster NMF + 4-Architecture 5-Fold Benchmarks (`65.59%` frozen real test Macro F1) + 7/7 Integrity Checks
5. **Tasks 10–11** `TfidfCosineIssueSolutionLinker` (`859` links) + Ranked Community Solution Themes + Quantitative Unresolved Criteria
6. **Research Q8** Cross-IIM (`19` IIMs) & Cross-Language behavioral contrasts + Live Multilingual Comment Analyzer

In [ ]:
import json, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'backend').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'backend'))

from app.api.demo import _data, _summary_data
from app.services.nlp.indian_reddit_lid import detect_indian_reddit_language
from app.services.nlp.preprocessing.emoji_normalizer import extract_emojis, normalize_emojis_semantic_affect
from app.services.nlp.preprocessing.slang_normalizer import detect_slang_terms, normalize_slang
from app.services.nlp.community_intelligence import (
    analyze_live_comment,
    build_cross_iim_comparison,
    build_cross_language_comparison,
    build_unresolved_issue_matrix,
)

store = _data()
summary = _summary_data()
ev = store['evaluation_raw']
ci = store['community_intelligence']
print(f"Loaded {store['metadata']['dataset_id']} v{store['metadata']['dataset_version']} | "
      f"{len(store['posts'])} threads | {len(store['comments'])} comments | synthetic={store['synthetic_only']}")

## 1. Tasks 5 & 6 — Multilingual LID, Gen-Z Slang & Emoji Semantic Affect Demo (PDF Page 1 Examples)

In [ ]:
pdf_examples = [
    'Bhai IIM ABC ka summer placement scene kaisa hai 😭',
    'ଆଜି କ୍ଲାସ୍ ବହୁତ hectic ଥିଲା bro',
    'IIM e giye ki actually ROI pawa jay?',
    'Bro hostel life 💀... mess food toh literally mid... but seniors helped a lot ngl.',
    'IIM jaana worth it hai kya?',
]
for text in pdf_examples:
    lid = detect_indian_reddit_language(text)
    norm = normalize_slang(normalize_emojis_semantic_affect(text))
    print(f"TEXT : {text}")
    print(f"  LID: lang={lid['language']} | script={lid['script']} | code_mixed={lid['is_code_mixed']}")
    print(f"  EMO: {[(e['emoji'], e['semantic_affect']) for e in extract_emojis(text)]} | SLANG: {[(s['term'], s['action']) for s in detect_slang_terms(text)]}")
    print(f"  NORM: {norm}\n")

In [ ]:
# Empirical Task 5 (langdetect failure study) & Task 6 (5-fold OOF preprocessing ablation)
abl = ci['task5_task6_ablations']
t5 = abl['task5_lid_study']
print('=== TASK 5: EXISTING-LID LIMITATION STUDY ===')
print(f"langdetect bogus non-Indian tags overall : {t5['langdetect_bogus_non_indian_total']}/{t5['corpus_size']} ({t5['langdetect_bogus_non_indian_rate']*100:.1f}%)")
print(f"langdetect bogus tags on code-mixed/Indic: {t5['langdetect_bogus_on_indic_count']}/{t5['code_mixed_or_roman_indic_count']} ({t5['langdetect_bogus_on_indic_rate']*100:.1f}%)")
print(f"12-probe accuracy: IndianRedditLID={t5['controlled_probe_accuracy']['indian_reddit_lid_accuracy']*100:.1f}% vs langdetect={t5['controlled_probe_accuracy']['langdetect_accuracy']*100:.1f}%\n")

print('=== TASK 6: 5-FOLD OOF PREPROCESSING ABLATION (n=367, zero test leakage) ===')
t6 = abl['task6_preprocessing_ablation']
for strat in ('raw', 'conventional_nlp', 'emoji_removed', 'emoji_token_slang', 'emoji_affect_slang'):
    tf = t6['tfidf_5task_ablation'][strat]['mean_5task_oof_f1']
    mu = t6['muril_wwm_affect_ablation'][strat]
    print(f"  {strat:20s} | TF-IDF 5-Task={tf*100:5.2f}% | MuRIL Sent={mu['sentiment']['oof_macro_f1']*100:5.2f}% | MuRIL Emo={mu['emotion']['oof_macro_f1']*100:5.2f}% | Affect Mean={mu['mean_affect_oof_f1']*100:5.2f}%")

## 2. Strictly Inductive 4-Architecture 5-Fold Benchmarks & 7/7 Pipeline Integrity Verifier

In [ ]:
print('=== 4-ARCHITECTURE 5-FOLD BENCHMARK (UNTOUCHED FROZEN TEST n=92, 0/44 DAPT OVERLAP) ===')
for k, v in ev['architecture_comparison']['aggregate'].items():
    print(f"  {v['label']:56s} | Test F1={v['mean_test_macro_f1']*100:5.2f}% | ECE={v['mean_calibration_ece']:.3f} | {v['mean_inference_ms']:.2f} ms/txt")

print('\n=== PER-TASK V6.1 DEPLOYED MODELS ===')
for t in ('issue', 'solution', 'sentiment', 'response', 'emotion'):
    row = ev['architecture_comparison']['per_task'][t]['production_hybrid']
    print(f"  {t:10s} | Test F1={row['test_macro_f1']*100:5.2f}% | Calib ECE={row['calibration_ece']:.3f} | Backbone={row['deployed_backbone']}")

ver = ev['benchmark_verification']
print(f"\nPipeline Integrity Verifier: verified={ver['verified']} ({ver['passed_checks']}/{ver['total_checks']} checks passed)")

## 3. Tasks 10 & 11 — Issue–Solution Linking, Ranked Community Solutions & Quantitative Unresolved Matrix

In [ ]:
ls = ci['linker_summary']
print(f"TfidfCosineIssueSolutionLinker: {ls['total_asserted_links']} links | {ls['unique_linked_solutions']} unique solutions across {ls['threads_with_linked_solutions']}/365 threads (mean blend={ls['mean_link_score']:.3f})\n")

um = build_unresolved_issue_matrix(store['posts'], store['comments'], ci)
print('=== TASK 11: QUANTITATIVE RESOLUTION & UNRESOLVED MATRIX ===')
for r in um['rows']:
    print(f"  {r['issue_label']:36s} | Vol={r['comment_mentions']:4d} | Sol={r['solution_density']*100:4.1f}% | Linked={r['linked_thread_coverage']*100:4.1f}% | Ver={r['verified_outcomes']:2d} | Neg={r['sentiment_shares']['negative']*100:4.1f}% | {r['resolution_strength']:8s} | {r['status']}")

print('\n=== TASK 10: TOP COMMUNITY SOLUTION THEMES (PLACEMENTS & FINANCIAL CONCERNS) ===')
for key in ('placements', 'financial_concerns', 'administrative_communication'):
    block = ci['issue_solution_clusters'][key]
    print(f"\n[{block['issue_label']}] ({block['linked_solution_count']} cosine-linked solutions):")
    for idx, c in enumerate(block['clusters'][:3], 1):
        print(f"  {idx}. {c['theme']} — {c['count']} linked ({c['share']*100:.0f}%)")

## 4. Research Question #8 — Cross-IIM, Cross-Language & Live Multilingual Analyzer

In [ ]:
print('=== CROSS-LANGUAGE REGISTER CONTRAST ===')
cl = build_cross_language_comparison(store['comments'])
for r in cl['registers']:
    print(f"  {r['label']:28s} | n={r['count']:4d} ({r['share']*100:4.1f}%) | Neg={r['sentiment']['negative']*100:4.1f}% | Sol={r['solution_rate']*100:4.1f}% | Emoji={r['emoji_rate']*100:4.1f}%")

print('\n=== TOP 8 MENTIONED IIMs ===')
for r in build_cross_iim_comparison(store['posts'], store['comments'])['iims'][:8]:
    print(f"  {r['iim']:18s} | Mentions={r['direct_comment_mentions']:3d} | Top Issue={r['top_issue_label']:35s} | Neg={r['sentiment']['negative']*100:4.1f}% | Sol={r['solution_rate']*100:4.1f}%")

print('\n=== LIVE ZERO-GPU ANALYZER DEMO ===')
demo_out = analyze_live_comment('Bro hostel life 💀... mess food toh literally mid... try food court near campus ngl', store['comments'], store['nn_index'])
print(json.dumps({'language': demo_out['language_detection'], 'predictions': {k: v.get('label', v.get('labels', v.get('is_solution'))) for k, v in demo_out['predictions'].items()}}, indent=2))